# MCQ Generator — Schema-Driven, Batch-Based, Judge-Validated

Generates MCQs from a schema — **no separate planning stage**.

**Two inputs:**
1. `InputSchema` — topic, subtopics with per-difficulty counts (Easy/Medium/Hard)
2. `ExampleQuestionSet` — reference questions that guide style and format per difficulty

**Flow:**
```
Schema + Examples
       ↓
EasyMCQAgent   → generate batch of 5 → DifficultyJudge → RubricJudge → QuestionStore
MediumMCQAgent → generate batch of 5 → DifficultyJudge → RubricJudge → QuestionStore
HardMCQAgent   → generate batch of 5 → DifficultyJudge → RubricJudge → QuestionStore
       ↓
MCQGenerationResult (store + rejected + warnings)
```

Run all cells top-to-bottom. Edit Cell 10 to change the schema and example questions.

## Cell 1 — Setup & Imports

In [1]:
import sys, os, json, warnings
from pathlib import Path
from types import SimpleNamespace
from typing import Literal

warnings.filterwarnings('ignore')

# ── Locate project root (directory that contains utils.py) ──────────────────
_candidate = Path().resolve()
if (_candidate / 'utils.py').exists():
    PROJECT_ROOT = _candidate
elif (_candidate.parent / 'utils.py').exists():
    PROJECT_ROOT = _candidate.parent
else:
    raise RuntimeError(
        f'Cannot locate project root from {_candidate}. '
        'Open Jupyter from d:/Topin or d:/Topin/notebooks.'
    )

DATA_DIR      = PROJECT_ROOT / 'data'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Auto-inject venv site-packages ──────────────────────────────────────────
_venv_sp = PROJECT_ROOT / '.venv' / 'Lib' / 'site-packages'
if not _venv_sp.exists():
    _venv_sp = PROJECT_ROOT / '.venv' / 'lib' / 'site-packages'
if _venv_sp.exists() and str(_venv_sp) not in sys.path:
    sys.path.insert(0, str(_venv_sp))
    print(f'Injected venv site-packages: {_venv_sp}')

import dspy
from pydantic import BaseModel, Field

print(f'Project root : {PROJECT_ROOT}')
print(f'Data dir     : {DATA_DIR}')
print(f'Artifacts    : {ARTIFACTS_DIR}')
print(f'DSPy version : {dspy.__version__}')

Injected venv site-packages: D:\Topin\.venv\Lib\site-packages
Project root : D:\Topin
Data dir     : D:\Topin\data
Artifacts    : D:\Topin\artifacts
DSPy version : 3.1.3


## Cell 2 — Configure DSPy (Mistral)

In [2]:
from utils import configure_dspy_from_env

lm = configure_dspy_from_env()
print(f'Active LM : {lm}')

Active LM : <dspy.clients.lm.LM object at 0x00000127DC4373D0>


## Cell 3 — Input Models

- `SubtopicRequirement` — per-subtopic counts for Easy / Medium / Hard
- `InputSchema` — topic + list of subtopic requirements + generation constraints
- `ExampleQuestion` — one reference question with optional `difficulty` and `subtopic` tags
- `ExampleQuestionSet` — collection with `filter_examples()` to pick relevant references per batch

In [3]:
class SubtopicRequirement(BaseModel):
    """Per-subtopic question counts for each of the 6 CEFR levels.

    CEFR mapping:
        A1, A2  →  Easy
        B1, B2  →  Medium
        C1, C2  →  Hard
    """
    subtopic:  str

    # ── Easy band ─────────────────────────────────────────────────────────────
    a1_count:  int = 0
    a2_count:  int = 0

    # ── Medium band ───────────────────────────────────────────────────────────
    b1_count:  int = 0
    b2_count:  int = 0

    # ── Hard band ─────────────────────────────────────────────────────────────
    c1_count:  int = 0
    c2_count:  int = 0

    # Convenience totals
    @property
    def easy_count(self) -> int:
        return self.a1_count + self.a2_count

    @property
    def medium_count(self) -> int:
        return self.b1_count + self.b2_count

    @property
    def hard_count(self) -> int:
        return self.c1_count + self.c2_count

    @property
    def total(self) -> int:
        return self.easy_count + self.medium_count + self.hard_count


class GenerationConstraints(BaseModel):
    questions_per_iteration:       int = 5   # batch size per generator call
    max_iterations_per_difficulty: int = 20  # max retry loops before giving up


class InputSchema(BaseModel):
    topic:       str
    subtopics:   list[SubtopicRequirement]
    constraints: GenerationConstraints = Field(default_factory=GenerationConstraints)


class ExampleQuestion(BaseModel):
    instruction:    str
    question:       str
    options:        list[str]
    correct_answer: str
    explanation:    str
    difficulty:     str | None = None   # 'Easy', 'Medium', or 'Hard'
    cefr:           str | None = None   # 'A1', 'A2', 'B1', 'B2', 'C1', 'C2'
    subtopic:       str | None = None


class ExampleQuestionSet(BaseModel):
    items: list[ExampleQuestion] = Field(default_factory=list)

    def filter_examples(
        self,
        *,
        subtopic: str,
        difficulty: str,
        cefr: str | None = None,
    ) -> list[ExampleQuestion]:
        """Return examples most relevant to this subtopic + difficulty/CEFR.
        Priority 1: match subtopic + exact CEFR level.
        Priority 2: match subtopic + difficulty band.
        Priority 3: match difficulty band only.
        Priority 4: return all (fallback).
        """
        if cefr:
            p1 = [q for q in self.items
                  if q.cefr == cefr and q.subtopic in (None, subtopic)]
            if p1:
                return p1

        p2 = [q for q in self.items
              if q.difficulty in (None, difficulty)
              and q.subtopic in (None, subtopic)]
        if p2:
            return p2

        p3 = [q for q in self.items if q.difficulty in (None, difficulty)]
        if p3:
            return p3

        return self.items


print('Input models defined.')
print('  SubtopicRequirement: a1_count, a2_count, b1_count, b2_count, c1_count, c2_count')
print('  CEFR mapping: A1/A2=Easy | B1/B2=Medium | C1/C2=Hard')

Input models defined.
  InputSchema: topic + list[SubtopicRequirement] + GenerationConstraints
  ExampleQuestionSet: filter_examples(subtopic, difficulty) → relevant examples


## Cell 4 — Output Models

- `MCQItem` — one accepted question with all fields
- `QuestionStore` — Pydantic model grouping accepted questions by difficulty (easy/medium/hard)
- `MCQGenerationResult` — final output: store + rejected records + warnings

In [4]:
class MCQItem(BaseModel):
    question_number:   int
    topic:             str
    subtopic:          str | None
    target_cefr:       str
    target_difficulty: str
    instruction:       str
    question:          str
    options:           list[str]    # exactly 4
    correct_answer:    str
    explanation:       str


class QuestionStore(BaseModel):
    """Accepted questions grouped by difficulty band."""
    easy:   list[MCQItem] = Field(default_factory=list)
    medium: list[MCQItem] = Field(default_factory=list)
    hard:   list[MCQItem] = Field(default_factory=list)

    def add(self, item: MCQItem) -> None:
        d = item.target_difficulty.strip().lower()
        if d == 'easy':
            self.easy.append(item)
        elif d == 'medium':
            self.medium.append(item)
        elif d == 'hard':
            self.hard.append(item)
        else:
            raise ValueError(f'Unknown difficulty: {item.target_difficulty}')

    def get_used_questions(self) -> list[str]:
        """All accepted questions across all difficulties."""
        return [q.question for q in self.easy + self.medium + self.hard]

    def count(self, difficulty: str) -> int:
        """Count accepted questions for a difficulty band (Easy / Medium / Hard)."""
        d = difficulty.strip().lower()
        if d == 'easy':   return len(self.easy)
        if d == 'medium': return len(self.medium)
        if d == 'hard':   return len(self.hard)
        raise ValueError(f'Unknown difficulty: {difficulty}')

    def count_by_cefr(self, cefr: str) -> int:
        """Count accepted questions for a specific CEFR level (A1/A2/B1/B2/C1/C2)."""
        return sum(1 for q in self.easy + self.medium + self.hard
                   if q.target_cefr == cefr)

    def all_items(self) -> list[MCQItem]:
        return self.easy + self.medium + self.hard


class MCQGenerationResult(BaseModel):
    store:    QuestionStore
    rejected: list[dict] = Field(default_factory=list)
    warnings: list[str]  = Field(default_factory=list)


print('Output models defined.')
print('  QuestionStore: easy / medium / hard lists')
print('  QuestionStore.count_by_cefr("A1") counts accepted A1 questions')

Output models defined.
  QuestionStore: easy / medium / hard lists
  MCQGenerationResult: store + rejected + warnings


## Cell 5 — MCQ Generator Signature + Agent

**Typed batch approach:**
- Input: `GenerationRequest` (topic, CEFR, filtered examples, used stems, batch size)
- Output: `GenerationBatch` → `list[GeneratedQuestion]`

One generator call produces a full batch of questions (up to `batch_size`).

In [5]:
class GenerationRequest(BaseModel):
    topic:              str
    subtopic:           str
    target_cefr:        str          # specific CEFR level: A1/A2/B1/B2/C1/C2
    target_difficulty:  str          # Easy / Medium / Hard
    example_questions:  list[ExampleQuestion]   # filtered reference examples
    already_used_questions: list[str]    # accepted questions so far — avoid repeating
    batch_size:         int          # number of questions to generate this iteration


class GeneratedQuestion(BaseModel):
    instruction:    str
    question:       str
    options:        list[str]        # exactly 4
    correct_answer: str
    explanation:    str


class GenerationBatch(BaseModel):
    questions: list[GeneratedQuestion]   # len == batch_size


class MCQGeneratorSignature(dspy.Signature):
    """Generate a batch of high-quality MCQ questions for the given topic and difficulty.

    Use the example_questions as a reference for style, vocabulary level, and format.
    Easy examples show A1/A2-level vocabulary; Medium show B1/B2; Hard show C1/C2.

    STRICT RULES — every question in the batch must follow these:
    - EXACTLY 4 options — no more, no less
    - correct_answer must be copied VERBATIM from one of the options
    - Each question must be different from all questions in already_used_questions
    - Only one answer can be correct; the other 3 are plausible but clearly wrong
    - Align vocabulary and grammar complexity to the target_cefr level
    - Avoid ambiguity — only one option can be correct

    Return exactly batch_size questions in the output.
    """
    request: GenerationRequest = dspy.InputField(
        desc='Batch generation parameters: topic, CEFR level, difficulty, reference examples, used questions'
    )
    output: GenerationBatch = dspy.OutputField(
        desc='Batch of generated MCQ questions, one per item in questions list'
    )


class MCQGeneratorAgent(dspy.Module):
    """Generates MCQs and runs the quota loop inside forward().

    Dependencies injected at construction time:
      self.store            — QuestionStore that accumulates accepted questions
      self.difficulty_judge — DifficultyJudgeWrapper (batch)
      self.rubric_judge     — RubricJudgeWrapper (batch)

    forward() only takes per-call parameters (topic, CEFR level, counts, etc.)
    and uses the injected dependencies via self.*.
    """

    def __init__(
        self,
        *,
        store:            QuestionStore,
        difficulty_judge: DifficultyJudgeWrapper,
        rubric_judge:     RubricJudgeWrapper,
    ):
        super().__init__()
        self.generate         = dspy.ChainOfThought(MCQGeneratorSignature)
        self.store            = store
        self.difficulty_judge = difficulty_judge
        self.rubric_judge     = rubric_judge

    def forward(
        self,
        *,
        schema:            InputSchema,
        example_questions: ExampleQuestionSet,
        topic:             str,
        subtopic:          str | None,
        target_cefr:       str,
        target_difficulty: str,
        target_count:      int,
        rejected:          list,
        warnings:          list,
    ) -> None:
        """Run the generation loop for one CEFR level until quota is met.

        Each iteration:
          1. self.generate(request) — ChainOfThought LLM call → GenerationBatch
          2. Hard-validate all questions in the batch
          3. self.difficulty_judge(valid_items) — batch LLM call
          4. self.rubric_judge(diff_passed)     — batch LLM call
          5. self.store.add(accepted)
          → Repeat until self.store.count_by_cefr(target_cefr) >= target_count
        """
        if target_count <= 0:
            return

        iteration = 0
        q_number  = len(self.store.all_items()) + 1

        print(f'\n[{target_difficulty} / {target_cefr}]  '
              f'target={target_count}  subtopic={subtopic!r}')

        while self.store.count_by_cefr(target_cefr) < target_count:
            iteration += 1
            if iteration > schema.constraints.max_iterations_per_difficulty:
                msg = (f'{target_difficulty}/{target_cefr}: max iterations '
                       f'({schema.constraints.max_iterations_per_difficulty}) reached. '
                       f'Accepted {self.store.count_by_cefr(target_cefr)}/{target_count}.')
                warnings.append(msg)
                print(f'  [WARNING] {msg}')
                break

            needed   = target_count - self.store.count_by_cefr(target_cefr)
            batch_sz = min(schema.constraints.questions_per_iteration, needed)
            relevant = example_questions.filter_examples(
                subtopic=subtopic or '',
                difficulty=target_difficulty,
                cefr=target_cefr,
            )
            request = GenerationRequest(
                topic=topic,
                subtopic=subtopic or '',
                target_cefr=target_cefr,
                target_difficulty=target_difficulty,
                example_questions=relevant,
                already_used_questions=self.store.get_used_questions(),
                batch_size=batch_sz,
            )

            # ── 1. Generate batch ─────────────────────────────────────────────
            print(f'  [Iter {iteration}] Generating {batch_sz} questions...')
            try:
                pred  = self.generate(request=request)
                batch = pred.output.questions
            except Exception as e:
                rejected.append({
                    'stage': 'generation', 'cefr': target_cefr,
                    'difficulty': target_difficulty,
                    'iteration': iteration, 'error': str(e),
                })
                print(f'  [Gen Error] iter={iteration}: {e}')
                continue

            # ── 2. Hard-validate entire batch ─────────────────────────────────
            valid_items: list[MCQItem] = []
            for q in batch:
                errors = hard_validate(q)
                if errors:
                    rejected.append({
                        'stage': 'hard_validate', 'cefr': target_cefr,
                        'difficulty': target_difficulty,
                        'iteration': iteration, 'question': q.question[:60], 'errors': errors,
                    })
                    print(f'  [Hard Fail] {errors}')
                    continue
                valid_items.append(MCQItem(
                    question_number=q_number,
                    topic=topic,
                    subtopic=subtopic,
                    target_cefr=target_cefr,
                    target_difficulty=target_difficulty,
                    instruction=q.instruction,
                    question=q.question,
                    options=q.options,
                    correct_answer=q.correct_answer,
                    explanation=q.explanation,
                ))
                q_number += 1

            if not valid_items:
                print(f'  [Skip] All {len(batch)} failed hard validation.')
                continue
            print(f'  [Hard]  {len(valid_items)}/{len(batch)} passed.')

            # ── 3. DifficultyJudge — one batch call ───────────────────────────
            try:
                diff_results = self.difficulty_judge(
                    items=valid_items,
                    expected_difficulty=target_difficulty,
                )
            except Exception as e:
                rejected.append({'stage': 'difficulty_judge_error', 'error': str(e)})
                print(f'  [Diff Error] {e}')
                continue

            diff_passed: list[MCQItem] = []
            for item, diff in zip(valid_items, diff_results):
                if diff.passed:
                    diff_passed.append(item)
                else:
                    rejected.append({
                        'stage': 'difficulty', 'q': item.question_number,
                        'cefr': target_cefr, 'difficulty': target_difficulty,
                        'reason': diff.reason, 'question': item.question[:60],
                    })
                    print(f'  [Diff Fail] Q{item.question_number}: {diff.reason}')

            print(f'  [Diff]  {len(diff_passed)}/{len(valid_items)} passed.')
            if not diff_passed:
                continue

            # ── 4. RubricJudge — one batch call ───────────────────────────────
            try:
                rub_results = self.rubric_judge(items=diff_passed)
            except Exception as e:
                rejected.append({'stage': 'rubric_judge_error', 'error': str(e)})
                print(f'  [Rub Error] {e}')
                continue

            accepted_count = 0
            for item, rub in zip(diff_passed, rub_results):
                if not rub.passed:
                    rejected.append({
                        'stage': 'rubric', 'q': item.question_number,
                        'cefr': target_cefr, 'difficulty': target_difficulty,
                        'reason': rub.reason, 'question': item.question[:60],
                    })
                    print(f'  [Rub  Fail] Q{item.question_number}: {rub.reason}')
                    continue

                # ── 5. Save to store ──────────────────────────────────────────
                self.store.add(item)
                accepted_count += 1
                total = self.store.count_by_cefr(target_cefr)
                print(f'  [Accepted]  Q{item.question_number} '
                      f'({target_cefr}/{target_difficulty}) '
                      f'{total}/{target_count}')
                if total >= target_count:
                    break

            print(f'  [Rubric] {accepted_count}/{len(diff_passed)} passed.')


print('MCQGeneratorAgent defined.')
print('  __init__(store, difficulty_judge, rubric_judge)  — injected dependencies')
print('  forward(schema, topic, target_cefr, target_count, ...)  — quota loop')
print()
print('  self.store            -> QuestionStore (accumulates accepted questions)')
print('  self.difficulty_judge -> DifficultyJudgeWrapper (batch)')
print('  self.rubric_judge     -> RubricJudgeWrapper (batch)')

MCQ Generator defined.
  Input  : request: GenerationRequest
  Output : output: GenerationBatch → list[GeneratedQuestion]


## Cell 6 — Load DifficultyJudge + RubricJudge

Copies the typed models from `difficulty_judge.ipynb` and `rubric_judge.ipynb` inline.  
Loads optimized agents from `artifacts/` if available, falls back to unoptimized.  
Wraps each judge to expose `.passed` and `.reason` for the per-difficulty agents.

In [6]:
# ── DifficultyJudge models (from difficulty_judge.ipynb) ─────────────────────
class Question(BaseModel):
    question_id:       str
    instruction:       str
    question:          str
    options:           list[str]
    correct_answer:    str
    explanation:       str
    target_cefr:       str
    target_difficulty: str


class DifficultyResult(BaseModel):
    question_id:          str
    predicted_cefr:       Literal['A1', 'A2', 'B1', 'B2', 'C1', 'C2']
    predicted_difficulty: Literal['Easy', 'Medium', 'Hard']


class DifficultyOutput(BaseModel):
    results: list[DifficultyResult]


class SimpleDifficultySignature(dspy.Signature):
    """Classify a list of MCQ questions by CEFR level and difficulty.
    For each question analyse vocabulary, grammar, reasoning load, and distractor difficulty.
    A1/A2 -> Easy | B1/B2 -> Medium | C1/C2 -> Hard.
    Return one DifficultyResult per question in the same order.
    """
    questions: list[Question]   = dspy.InputField(desc='List of MCQ questions to be classified')
    output:    DifficultyOutput = dspy.OutputField(desc='Classification results for all questions')


class SimpleDifficultyAgent(dspy.Module):
    def __init__(self):
        super().__init__()
        self.judge = dspy.ChainOfThought(SimpleDifficultySignature)
    def forward(self, questions: list[Question]) -> dspy.Prediction:
        return self.judge(questions=questions)


# ── RubricJudge models (from rubric_judge.ipynb) ─────────────────────────────
class RubricQuestion(BaseModel):
    question_id:       str
    instruction:       str
    question:          str
    options:           list[str]
    correct_answer:    str
    explanation:       str
    target_cefr:       str
    target_difficulty: str
    language_variant:  str


class RubricResult(BaseModel):
    question_id:                          str
    grammatical_accuracy:                 Literal['No Issues', 'Minor Issues', 'Major Issues']
    spelling:                             Literal['No Issues', 'Minor Issues', 'Major Issues']
    ambiguity:                            Literal['No Issue', 'Minor Issue', 'Major Issue']
    functionality_alignment:              Literal['Aligned', 'Partially Aligned', 'Not Aligned']
    instruction_clarity_appropriateness:  Literal['Clear', 'Needs Improvement', 'Unclear']
    academic_language_exam_acceptability: Literal['Acceptable', 'Needs Improvement', 'Not Acceptable']
    option_explanation_consistency:       Literal['Consistent', 'Inconsistent']
    readability:                          Literal['Good', 'Needs Improvement', 'Poor']
    formatting_spacing:                   Literal['No Issues', 'Minor Issues', 'Major Issues']
    punctuation:                          Literal['No Issues', 'Minor Issues', 'Major Issues']
    british_american_english_consistency: Literal['Consistent', 'Inconsistent']
    overall_decision:                     Literal['Pass', 'Revise', 'Fail']
    priority_reason:                      str
    revision_feedback:                    str


class RubricOutput(BaseModel):
    results: list[RubricResult]


class RubricJudgeSignature(dspy.Signature):
    """Evaluate a list of MCQ questions using the rubric.
    ambiguity is highest priority — Major Issue forces Fail.
    Return one RubricResult per question in the same order.
    """
    questions: list[RubricQuestion] = dspy.InputField(desc='List of MCQ questions to be evaluated')
    output:    RubricOutput         = dspy.OutputField(desc='Rubric evaluation results for all questions')


class RubricJudgeAgent(dspy.Module):
    def __init__(self):
        super().__init__()
        self.judge = dspy.ChainOfThought(RubricJudgeSignature)
    def forward(self, questions: list[RubricQuestion]) -> dspy.Prediction:
        return self.judge(questions=questions)


# ── Load optimized agents from artifacts ─────────────────────────────────────
DIFF_ARTIFACT = ARTIFACTS_DIR / 'simple_difficulty_optimized.json'
RUB_ARTIFACT  = ARTIFACTS_DIR / 'rubric_agent_optimized.json'

_diff_agent = SimpleDifficultyAgent()
if DIFF_ARTIFACT.exists():
    _diff_agent.load(str(DIFF_ARTIFACT))
    print(f'Loaded DifficultyJudge from: {DIFF_ARTIFACT}')
else:
    print(f'DifficultyJudge: using unoptimized agent (artifact not found: {DIFF_ARTIFACT})')

_rub_agent = RubricJudgeAgent()
if RUB_ARTIFACT.exists():
    _rub_agent.load(str(RUB_ARTIFACT))
    print(f'Loaded RubricJudge from   : {RUB_ARTIFACT}')
else:
    print(f'RubricJudge: using unoptimized agent (artifact not found: {RUB_ARTIFACT})')


# ── Batch judge wrappers ──────────────────────────────────────────────────────

class DifficultyJudgeWrapper:
    """Wraps SimpleDifficultyAgent.
    Accepts a batch of MCQItems, returns a list of SimpleNamespace(.passed, .reason).
    """

    def __call__(
        self,
        *,
        items: list,          # list[MCQItem]
        expected_difficulty: str,
    ) -> list:                # list[SimpleNamespace]
        questions = [
            Question(
                question_id=str(item.question_number),
                instruction=item.instruction,
                question=item.question,
                options=item.options,
                correct_answer=item.correct_answer,
                explanation=item.explanation,
                target_cefr=item.target_cefr,
                target_difficulty=item.target_difficulty,
            )
            for item in items
        ]
        pred = _diff_agent(questions=questions)
        results = []
        for res in pred.output.results:
            passed = res.predicted_difficulty.lower() == expected_difficulty.lower()
            results.append(SimpleNamespace(
                passed=passed,
                reason=f'predicted_cefr={res.predicted_cefr} predicted_difficulty={res.predicted_difficulty}',
            ))
        return results


class RubricJudgeWrapper:
    """Wraps RubricJudgeAgent.
    Accepts a batch of MCQItems, returns a list of SimpleNamespace(.passed, .reason).
    """

    def __init__(self, language_variant: str = 'British English'):
        self.language_variant = language_variant

    def __call__(self, *, items: list) -> list:   # list[MCQItem] -> list[SimpleNamespace]
        questions = [
            RubricQuestion(
                question_id=str(item.question_number),
                instruction=item.instruction,
                question=item.question,
                options=item.options,
                correct_answer=item.correct_answer,
                explanation=item.explanation,
                target_cefr=item.target_cefr,
                target_difficulty=item.target_difficulty,
                language_variant=self.language_variant,
            )
            for item in items
        ]
        pred = _rub_agent(questions=questions)
        return [
            SimpleNamespace(
                passed=res.overall_decision == 'Pass',
                reason=res.priority_reason,
            )
            for res in pred.output.results
        ]


difficulty_judge = DifficultyJudgeWrapper()
rubric_judge     = RubricJudgeWrapper(language_variant='British English')

print('\nBatch judge wrappers ready.')
print('  difficulty_judge(items=[...], expected_difficulty="Easy") -> list of .passed/.reason')
print('  rubric_judge(items=[...])                                 -> list of .passed/.reason')

Loaded DifficultyJudge from: D:\Topin\artifacts\simple_difficulty_optimized.json
Loaded RubricJudge from   : D:\Topin\artifacts\rubric_agent_optimized.json

Judge wrappers ready.
  difficulty_judge(item=..., expected_difficulty=...) → .passed + .reason
  rubric_judge(item=...)                              → .passed + .reason


## Cell 7 — Hard Validate Helper

Structural checks before sending to judges:
- Exactly 4 options
- `correct_answer` is one of the options
- `explanation` is not empty

In [7]:
def hard_validate(q: GeneratedQuestion) -> list[str]:
    """Returns a list of error strings. Empty list = valid."""
    errors = []
    if not isinstance(q.options, list) or len(q.options) != 4:
        errors.append(f'Expected 4 options, got {len(q.options) if isinstance(q.options, list) else type(q.options).__name__}')
    if q.correct_answer not in q.options:
        errors.append(f'correct_answer "{q.correct_answer}" not found in options')
    if not q.explanation or not q.explanation.strip():
        errors.append('explanation is empty')
    if not q.instruction or not q.instruction.strip():
        errors.append('instruction is empty')
    if not q.question or not q.question.strip():
        errors.append('question is empty')
    return errors


print('hard_validate() defined.')
print('  Checks: 4 options | correct_answer in options | explanation | instruction | question not empty')

hard_validate() defined.
  Checks: 4 options | correct_answer in options | explanation not empty | stem not empty


## Cell 8 — Per-Difficulty Agents

- `BaseDifficultyAgent` — shared generation loop logic
- `EasyMCQAgent`   — `difficulty_name='Easy'`,   `default_cefr='A2'`
- `MediumMCQAgent` — `difficulty_name='Medium'`, `default_cefr='B1'`
- `HardMCQAgent`   — `difficulty_name='Hard'`,   `default_cefr='B2'`

Each agent loops, generates a batch of 5, validates via judges, stores accepted questions.

In [8]:
_CEFR_TO_DIFFICULTY = {
    'A1': 'Easy',  'A2': 'Easy',
    'B1': 'Medium', 'B2': 'Medium',
    'C1': 'Hard',  'C2': 'Hard',
}


class BaseDifficultyAgent:
    """Binds difficulty_name and delegates to MCQGeneratorAgent.forward().

    Judges and store are owned by MCQGeneratorAgent — not held here.
    """
    difficulty_name: str = ''
    default_cefr:    str = ''

    def __init__(self, *, generator: MCQGeneratorAgent):
        self.generator = generator

    def generate_questions(
        self,
        *,
        schema:            InputSchema,
        example_questions: ExampleQuestionSet,
        topic:             str,
        subtopic:          str | None,
        target_cefr:       str,
        target_count:      int,
        rejected:          list,
        warnings:          list,
    ) -> None:
        # self.generator(...) calls MCQGeneratorAgent.forward()
        self.generator(
            schema=schema,
            example_questions=example_questions,
            topic=topic,
            subtopic=subtopic,
            target_cefr=target_cefr,
            target_difficulty=self.difficulty_name,
            target_count=target_count,
            rejected=rejected,
            warnings=warnings,
        )


class EasyMCQAgent(BaseDifficultyAgent):
    difficulty_name = 'Easy'
    default_cefr    = 'A2'

class MediumMCQAgent(BaseDifficultyAgent):
    difficulty_name = 'Medium'
    default_cefr    = 'B1'

class HardMCQAgent(BaseDifficultyAgent):
    difficulty_name = 'Hard'
    default_cefr    = 'B2'


print('Per-difficulty agents defined.')
print('  __init__(generator)  — only holds MCQGeneratorAgent reference')
print('  generate_questions() — calls self.generator(...) -> MCQGeneratorAgent.forward()')

Per-difficulty agents defined.
  EasyMCQAgent   — difficulty=Easy,   default_cefr=A2
  MediumMCQAgent — difficulty=Medium, default_cefr=B1
  HardMCQAgent   — difficulty=Hard,   default_cefr=B2


## Cell 9 — Orchestrator

`MCQGenerationOrchestrator.run()` iterates over each `SubtopicRequirement` in the schema  
and runs Easy → Medium → Hard agents in sequence, collecting all results.

In [9]:
_CEFR_LEVELS = ['A1', 'A2', 'B1', 'B2', 'C1', 'C2']
_CEFR_TO_DIFFICULTY = {
    'A1': 'Easy',  'A2': 'Easy',
    'B1': 'Medium', 'B2': 'Medium',
    'C1': 'Hard',  'C2': 'Hard',
}


class MCQGenerationOrchestrator:
    """Creates and wires all components, then runs the generation loop.

    Ownership:
      self.store     — QuestionStore (shared by all agents via MCQGeneratorAgent)
      self.generator — MCQGeneratorAgent (owns store + judges)
      self.easy/medium/hard_agent — per-difficulty wrappers (hold only generator ref)
    """

    def __init__(
        self,
        *,
        difficulty_judge: DifficultyJudgeWrapper,
        rubric_judge:     RubricJudgeWrapper,
    ):
        self.store = QuestionStore()

        self.generator = MCQGeneratorAgent(
            store=self.store,
            difficulty_judge=difficulty_judge,
            rubric_judge=rubric_judge,
        )

        self.easy_agent   = EasyMCQAgent(generator=self.generator)
        self.medium_agent = MediumMCQAgent(generator=self.generator)
        self.hard_agent   = HardMCQAgent(generator=self.generator)

        self._agents = {
            'Easy':   self.easy_agent,
            'Medium': self.medium_agent,
            'Hard':   self.hard_agent,
        }

    def run(
        self,
        *,
        schema:            InputSchema,
        example_questions: ExampleQuestionSet,
    ) -> MCQGenerationResult:
        rejected = []
        warnings = []

        for req in schema.subtopics:
            SEP = '=' * 60
            print('\n' + SEP)
            print(f'Topic: {schema.topic}  |  Subtopic: {req.subtopic}')
            print(f'  Easy:   A1={req.a1_count}  A2={req.a2_count}')
            print(f'  Medium: B1={req.b1_count}  B2={req.b2_count}')
            print(f'  Hard:   C1={req.c1_count}  C2={req.c2_count}')
            print(SEP)

            for cefr in _CEFR_LEVELS:
                count_attr   = cefr.lower() + '_count'
                target_count = getattr(req, count_attr)
                if target_count == 0:
                    continue

                difficulty = _CEFR_TO_DIFFICULTY[cefr]
                agent      = self._agents[difficulty]

                agent.generate_questions(
                    schema=schema,
                    example_questions=example_questions,
                    topic=schema.topic,
                    subtopic=req.subtopic,
                    target_cefr=cefr,
                    target_count=target_count,
                    rejected=rejected,
                    warnings=warnings,
                )

        return MCQGenerationResult(
            store=self.store,
            rejected=rejected,
            warnings=warnings,
        )


print('MCQGenerationOrchestrator defined.')
print('  __init__(difficulty_judge, rubric_judge)')
print('    -> creates QuestionStore')
print('    -> creates MCQGeneratorAgent(store, difficulty_judge, rubric_judge)')
print('    -> creates EasyMCQAgent / MediumMCQAgent / HardMCQAgent(generator)')

MCQGenerationOrchestrator defined.
  orchestrator.run(schema=..., example_questions=...) → MCQGenerationResult


## Cell 10 — Define Schema + Examples, Then Run

**Edit this cell** to change the topic, subtopics, counts, and reference examples.

- `easy_count` / `medium_count` / `hard_count` — how many questions to generate per difficulty
- `easy_cefr` / `medium_cefr` / `hard_cefr` — target CEFR level per difficulty
- `ExampleQuestion` items — reference questions showing the expected style and format

In [10]:
# ── Define schema ─────────────────────────────────────────────────────────────
schema = InputSchema(
    topic='English Grammar',
    subtopics=[
        SubtopicRequirement(
            subtopic='Present Tense',
            a1_count=1,    # change to e.g. 10 for a full run
            a2_count=1,
            b1_count=1,
            b2_count=1,
            c1_count=1,
            c2_count=1,
        )
    ],
    constraints=GenerationConstraints(
        questions_per_iteration=5,
        max_iterations_per_difficulty=10,
    ),
)

# ── Define reference example questions (one per CEFR level) ──────────────────
example_questions = ExampleQuestionSet(items=[
    ExampleQuestion(
        instruction='Read the sentence and choose the correct word.',
        question='She _______ to school every day.',
        options=['go', 'goes', 'going', 'gone'],
        correct_answer='goes',
        explanation='Third-person singular subjects require -s/-es in simple present.',
        difficulty='Easy', cefr='A1', subtopic='Present Tense',
    ),
    ExampleQuestion(
        instruction='Read the sentence and choose the correct word.',
        question='My brother _______ football every weekend with his friends.',
        options=['play', 'plays', 'playing', 'played'],
        correct_answer='plays',
        explanation="Third-person singular 'brother' requires the -s form.",
        difficulty='Easy', cefr='A2', subtopic='Present Tense',
    ),
    ExampleQuestion(
        instruction='Read the sentence and identify the error.',
        question="Spot the error: 'They is playing football in the garden right now.'",
        options=['They -> It', 'is -> are', 'playing -> play', 'in -> at'],
        correct_answer='is -> are',
        explanation="Plural subject 'they' requires 'are' in present continuous.",
        difficulty='Medium', cefr='B1', subtopic='Present Tense',
    ),
    ExampleQuestion(
        instruction='Read the options and choose the correct sentence.',
        question='Which sentence correctly uses the present perfect to emphasise recent completion?',
        options=[
            'She just finished the report.',
            'She has just finished the report.',
            'She is just finishing the report.',
            'She had just finished the report.',
        ],
        correct_answer='She has just finished the report.',
        explanation="Present perfect 'has finished' emphasises recent completion.",
        difficulty='Medium', cefr='B2', subtopic='Present Tense',
    ),
    ExampleQuestion(
        instruction='Read the options and choose the correct sentence.',
        question='Which sentence demonstrates correct subject-verb agreement in a complex noun phrase?',
        options=[
            'The quality of the reports are improving steadily.',
            'The quality of the reports is improving steadily.',
            'The quality of the reports have improved steadily.',
            'The quality of the reports were improving steadily.',
        ],
        correct_answer='The quality of the reports is improving steadily.',
        explanation="The head noun 'quality' is singular, so the verb must be singular.",
        difficulty='Hard', cefr='C1', subtopic='Present Tense',
    ),
    ExampleQuestion(
        instruction='Read the options and choose the most formal sentence.',
        question='In which sentence is the present tense used to convey a scheduled future event with the highest degree of formality?',
        options=[
            'The summit commences at 09:00 tomorrow.',
            'The summit will commence at 09:00 tomorrow.',
            'The summit is commencing at 09:00 tomorrow.',
            'The summit is going to commence at 09:00 tomorrow.',
        ],
        correct_answer='The summit commences at 09:00 tomorrow.',
        explanation='Simple present for scheduled events is the most formal register.',
        difficulty='Hard', cefr='C2', subtopic='Present Tense',
    ),
])

# ── Build orchestrator and run ────────────────────────────────────────────────
# Orchestrator creates QuestionStore + MCQGeneratorAgent internally.
# Only the judges (already loaded in Cell 6) are passed in.
orchestrator = MCQGenerationOrchestrator(
    difficulty_judge=difficulty_judge,
    rubric_judge=rubric_judge,
)

result = orchestrator.run(schema=schema, example_questions=example_questions)

# ── Summary ───────────────────────────────────────────────────────────────────
SEP = '=' * 60
print()
print(SEP)
print('GENERATION COMPLETE')
print(SEP)

for req in schema.subtopics:
    print(f'\nSubtopic: {req.subtopic}')
    for cefr in _CEFR_LEVELS:
        count_attr = cefr.lower() + '_count'
        target     = getattr(req, count_attr)
        if target == 0:
            continue
        accepted = result.store.count_by_cefr(cefr)
        status   = 'OK' if accepted >= target else 'PARTIAL'
        diff     = _CEFR_TO_DIFFICULTY[cefr]
        print(f'  {cefr} ({diff:<6}): {accepted}/{target}  [{status}]')

print(f'\nTotal accepted : {len(result.store.all_items())}')
print(f'Total rejected : {len(result.rejected)}')
if result.warnings:
    print('Warnings:')
    for w in result.warnings:
        print(f'  {w}')

print()
print(f'{"Q#":<4} {"CEFR":<6} {"Difficulty":<10} {"Stem (preview)"}')
print('-' * 72)
for item in result.store.all_items():
    print(f'{item.question_number:<4} {item.target_cefr:<6} '
          f'{item.target_difficulty:<10} {item.question[:48]}...')


Topic: English Grammar  |  Subtopic: Present Tense
  Easy=3 (A2)  Medium=2 (B1)  Hard=2 (B2)

[Easy] target=3  cefr=A2  subtopic=Present Tense
  [Accepted]  Q1 — Easy 1/3
  [Accepted]  Q2 — Easy 2/3
  [Hard Fail] ['correct_answer "went" not found in options']
  [Accepted]  Q3 — Easy 3/3

[Medium] target=2  cefr=B1  subtopic=Present Tense
  [Accepted]  Q4 — Medium 1/2
  [Accepted]  Q5 — Medium 2/2

[Hard] target=2  cefr=B2  subtopic=Present Tense
  [Accepted]  Q6 — Hard 1/2
  [Accepted]  Q7 — Hard 2/2

GENERATION COMPLETE

Subtopic: Present Tense
  Easy    : 3/3  [OK]
  Medium  : 2/2  [OK]
  Hard    : 2/2  [OK]

Total accepted : 7
Total rejected : 1

Q#   Difficulty CEFR   Stem (preview)
----------------------------------------------------------------------
1    Easy       A2     They _______ to the park every afternoon....
2    Easy       A2     He _______ to work at 8 o'clock every morning....
3    Easy       A2     They _______ to the cinema every weekend....
4    Medium     B1     

## Cell 11 — Save Output

In [11]:
output = {
    'schema': schema.model_dump(),
    'summary': {
        'easy':   {'accepted': result.store.count('Easy'),   'rejected': sum(1 for r in result.rejected if r.get('difficulty') == 'Easy')},
        'medium': {'accepted': result.store.count('Medium'), 'rejected': sum(1 for r in result.rejected if r.get('difficulty') == 'Medium')},
        'hard':   {'accepted': result.store.count('Hard'),   'rejected': sum(1 for r in result.rejected if r.get('difficulty') == 'Hard')},
    },
    'questions': {
        'easy':   [q.model_dump() for q in result.store.easy],
        'medium': [q.model_dump() for q in result.store.medium],
        'hard':   [q.model_dump() for q in result.store.hard],
    },
    'rejected': result.rejected,
    'warnings': result.warnings,
}

out_path = DATA_DIR / 'mcq' / 'mcq_generator_output.json'
out_path.write_text(json.dumps(output, indent=2, ensure_ascii=False), encoding='utf-8')

print(f'Saved to: {out_path}')
print(f'  Easy   : {len(result.store.easy)} questions')
print(f'  Medium : {len(result.store.medium)} questions')
print(f'  Hard   : {len(result.store.hard)} questions')
print(f'  Rejected: {len(result.rejected)}')

Saved to: D:\Topin\data\mcq_generator_output.json
  Easy   : 3 questions
  Medium : 2 questions
  Hard   : 2 questions
  Rejected: 1
